# Anatomy of a Flop — Preprocessing

## Scopul
Curățăm datele și le pregătim pentru modelare.

### Pași:
1. Eliminarea valorilor lipsă
2. Feature engineering
3. Salvarea datelor curate

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/movies_raw.csv')
print(f" Date încărcate: {df.shape}")

 Date încărcate: (1000, 18)


In [3]:
# Verificăm valorile lipsă înainte de curățare
missing_values = df.isnull().sum().sort_values(ascending=False)

print("Valori lipsă pe coloane:")

if missing_values[missing_values > 0].empty:
    print("Nu există valori lipsă în datasetul brut.")
else:
    print(missing_values[missing_values > 0])

Valori lipsă pe coloane:
Nu există valori lipsă în datasetul brut.


In [4]:
# Exercițiu: simulăm valori lipsă pentru variabila runtime
df_missing_demo = df.copy()

np.random.seed(42)
missing_indices = df_missing_demo.sample(frac=0.05, random_state=42).index

df_missing_demo.loc[missing_indices, 'runtime'] = np.nan

print("Valori lipsă simulate pentru runtime:")
print(df_missing_demo['runtime'].isnull().sum())

Valori lipsă simulate pentru runtime:
50


In [5]:
# Imputăm valorile lipsă simulate folosind mediana
runtime_median = df_missing_demo['runtime'].median()

df_missing_demo['runtime'] = df_missing_demo['runtime'].fillna(runtime_median)

print("Valori lipsă după imputare:")
print(df_missing_demo['runtime'].isnull().sum())

Valori lipsă după imputare:
0


Deoarece datasetul brut nu conține valori lipsă, am simulat artificial valori lipsă pentru variabila `runtime`, pentru a demonstra o metodă de imputare. Am folosit mediana, deoarece este mai robustă la valori extreme decât media.

In [6]:
# Eliminăm filmele fără date financiare
df_clean = df[(df['budget'] > 100000) & (df['revenue'] > 100000)].copy()
print(f" filme cu date complete: {len(df_clean)}")

# Disappointment Index
df_clean['expected_revenue'] = df_clean['budget'] * 2.5
df_clean['disappointment_index'] = df_clean['revenue'] / df_clean['expected_revenue']
df_clean['roi'] = (df_clean['revenue'] - df_clean['budget']) / df_clean['budget']
df_clean['release_year'] = pd.to_datetime(df_clean['release_date']).dt.year
df_clean['release_month'] = pd.to_datetime(df_clean['release_date']).dt.month

print(df_clean[['budget','revenue','disappointment_index','roi']].describe())

 filme cu date complete: 866
             budget       revenue  disappointment_index         roi
count  8.660000e+02  8.660000e+02            866.000000  866.000000
mean   8.310540e+07  3.606459e+08              3.005975    6.514936
std    7.746028e+07  3.805884e+08              5.531719   13.829298
min    3.250000e+05  1.243670e+05              0.000908   -0.997730
25%    2.000000e+07  8.902500e+07              0.984993    1.462482
50%    6.000000e+07  2.449706e+08              1.667067    3.167668
75%    1.300000e+08  5.047626e+08              2.809941    6.024851
max    4.899000e+08  2.923706e+09             86.474581  215.186452


### Observații după curățare

- **866 filme** au date financiare complete din 1000
- **DI mediu: 3.0** — în medie filmele câștigă de 3x față de așteptări
- **ROI mediu: 6.5** — returnul investiției e de 650% în medie
- **Std DI foarte mare (5.5)** — variații enorme între filme
- **Min ROI: -0.99** → există filme care au pierdut aproape tot bugetul

In [7]:
# Salvăm datele curate
df_clean.to_csv('../data/movies_clean.csv', index=False)
print(f" Salvat! {len(df_clean)} filme în movies_clean.csv")

 Salvat! 866 filme în movies_clean.csv


## 3. Tratarea Outlierilor
Identificăm outlierii folosind metoda IQR.

In [8]:
Q1 = df_clean['disappointment_index'].quantile(0.25)
Q3 = df_clean['disappointment_index'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df_clean[
    (df_clean['disappointment_index'] < lower) | 
    (df_clean['disappointment_index'] > upper)
]

print(f"Q1: {Q1:.2f}")
print(f"Q3: {Q3:.2f}")
print(f"IQR: {IQR:.2f}")
print(f"Lower bound: {lower:.2f}")
print(f"Upper bound: {upper:.2f}")
print(f"\nNumăr outlieri: {len(outliers)} filme")
print(f"\nTop 10 outlieri:")
print(outliers.nlargest(10, 'disappointment_index')[
    ['title', 'budget', 'revenue', 'disappointment_index']
].to_string(index=False))

Q1: 0.98
Q3: 2.81
IQR: 1.82
Lower bound: -1.75
Upper bound: 5.55

Număr outlieri: 86 filme

Top 10 outlieri:
                          title   budget   revenue  disappointment_index
                      Halloween   325000  70260597             86.474581
Snow White and the Seven Dwarfs  1488423 184925486             49.697025
                          Rocky  1000000 117253345             46.901338
             Gone with the Wind  4000000 402352579             40.235258
                     Cinderella  2900000 263600000             36.358621
                            Saw  1200000 104045735             34.681912
                  The Evil Dead   350000  29612367             33.842705
     E.T. the Extra-Terrestrial 10500000 797307407             30.373616
                      Star Wars 11000000 775398007             28.196291
                           Jaws  7000000 470653000             26.894457


### Rezultatele analizei outlierilor

Am identificat 86 filme (9.9% din dataset) cu DI > 5.55
care depasesc limita superioara calculata prin metoda IQR.

Acestea sunt filme cu bugete foarte mici care au generat
venituri de zeci de ori mai mari decat estimarile noastre —
Halloween (budget 325.000$, DI = 86), Rocky, Star Wars, Jaws.

Nu le eliminam din analiza principala deoarece reprezinta
exact fenomenul pe care il studiem — filmele care au
surprins cel mai mult industria. Le salvam insa separat
in movies_clean_no_outliers.csv pentru a putea compara
performanta modelelor cu si fara ele.